# Scope, Closures, and the Names Python Remembers
Some values vanish when a function ends, while others seem to linger inside closures long after the original call is over. That behavior starts to make sense once you look at cells, free variables, and the frames Python preserves.

When people struggle with **Scope, Closures, and the Names Python Remembers**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Use LEGB as an explanatory model rather than a slogan
- See closures as remembered references
- Understand `nonlocal` without fear
- Connect scope to function factories and decorators


A closure keeps access to objects referenced from the enclosing scope. What is remembered is not “the text of the variable”, but the live binding relationship needed by the inner function.


In [1]:
def make_counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc

c = make_counter()
print(c(), c(), c())
print(c.__closure__)


1 2 3
(<cell at 0x000002CE83597F40: int object at 0x00007FFAF014D368>,)


Disassembly can reveal free variables and closure-related behavior. You do not need to become a compiler engineer; you only need to notice that nested functions carry extra runtime structure.


In [2]:
import dis

def outer():
    x = 10
    def inner():
        return x + 1
    return inner

dis.dis(outer)


              0 MAKE_CELL                1 (x)

  3           2 RESUME                   0

  4           4 LOAD_CONST               1 (10)
              6 STORE_DEREF              1 (x)

  5           8 LOAD_CLOSURE             1 (x)
             10 BUILD_TUPLE              1
             12 LOAD_CONST               2 (<code object inner at 0x000002CE836006B0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_6352\2315492634.py", line 5>)
             14 MAKE_FUNCTION            8 (closure)
             16 STORE_FAST               0 (inner)

  7          18 LOAD_FAST                0 (inner)
             20 RETURN_VALUE

Disassembly of <code object inner at 0x000002CE836006B0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_6352\2315492634.py", line 5>:
              0 COPY_FREE_VARS           1

  5           2 RESUME                   0

  6           4 LOAD_DEREF               0 (x)
              6 LOAD_CONST               1 (1)
              8 BINARY_OP                0 (+)
  

Python checks local, enclosing, global, then built-in names.

That is why inner functions can still use outer names later.

It says “when I assign, I mean the enclosing binding, not a fresh local one”.

They are not only theory. They help with stateful callbacks, factories, and decorators.


The name is found in the enclosing scope because it is not local to the inner function.


In [3]:
def outer():
    msg = "from outer"
    def inner():
        return msg
    return inner()

print(outer())


from outer


The returned function keeps the outer value around.


In [4]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor
    return multiply

double = make_multiplier(2)
print(double(10))


20


Without `nonlocal`, this rebinding would create a new local variable instead.


In [5]:
def make_total():
    total = 0
    def add(value):
        nonlocal total
        total += value
        return total
    return add

add = make_total()
print(add(5), add(7))


5 12


This is not only a classroom topic. **Scope, Closures, and the Names Python Remembers** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Memorizing LEGB letters without really understanding lookup
- Using globals when a closure would keep state cleaner
- Forgetting that `nonlocal` is about rebinding, not only reading


- What is a closure?
- What does `nonlocal` do?
- Why can a nested function still see an outer name after the outer function returns?


- Closures are remembered access to enclosing names.
- LEGB is a useful runtime search order.
- `nonlocal` controls rebinding of enclosing names.


If this notebook did its job, **Scope, Closures, and the Names Python Remembers** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Scope, Closures, and the Names Python Remembers is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Scope, Closures, and the Names Python Remembers, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x000002CE83427D00, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_6352\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST         

The deepest gain here is to inspect closure cells directly. That turns ?the function remembers a variable? from a vague sentence into an inspectable runtime fact. Inner functions do not keep magic copies of outer variables. They keep access paths through closure cells.


In [7]:
def outer():
    x = 10
    y = 20
    def inner(z):
        return x + y + z
    return inner

fn = outer()
print('freevars:', fn.__code__.co_freevars)
print('cellvars in outer not visible here, but closure cells are:', fn.__closure__)
print('cell contents:', [cell.cell_contents for cell in fn.__closure__])


freevars: ('x', 'y')
cellvars in outer not visible here, but closure cells are: (<cell at 0x000002CE83597CD0: int object at 0x00007FFAF014D448>, <cell at 0x000002CE835E46D0: int object at 0x00007FFAF014D588>)
cell contents: [10, 20]


The strongest takeaway here is that scope is not only about where names are visible. It is about where Python will look next, what bindings survive, and how nested functions can keep those bindings alive. That makes scope one of the bridges between beginner Python and serious runtime understanding. Closures are especially important because they show how functions can carry state without classes or globals.


## Final Takeaway

The deepest way to revise **Scope, Closures, and the Names Python Remembers** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
